<a href="https://colab.research.google.com/github/reza-pishva/RNN-projects/blob/main/clustering_lube_oil_g11_final_result.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
import warnings # Ignore specific warnings
warnings.filterwarnings("ignore")
from sklearn.neighbors import NearestNeighbors # Import NearestNeighbors

In [2]:
df1 = pd.read_excel('output_lube_oil_g11.xlsx')

In [3]:
from sklearn.preprocessing import StandardScaler

# انتخاب ستون‌ها برای استانداردسازی
data_to_scale = df1[['AssetID_8341', 'AssetID_8342', 'AssetID_8343', 'AssetID_8344',
       'AssetID_8346', 'AssetID_9286', 'AssetID_9287']]

# استانداردسازی داده‌ها
scaler = StandardScaler()
scaled_data = scaler.fit_transform(data_to_scale)

# تبدیل خروجی به دیتافریم با همان نام ستون‌ها
scaled_df = pd.DataFrame(scaled_data, columns=['AssetID_8341', 'AssetID_8342', 'AssetID_8343', 'AssetID_8344',
       'AssetID_8346', 'AssetID_9286', 'AssetID_9287'])

In [4]:
scaled_df_clean = scaled_df.dropna()

In [5]:
# اجرای DBSCAN روی داده‌های استانداردشده
dbscan = DBSCAN(eps=0.6, min_samples=9)
labels = dbscan.fit_predict(scaled_df_clean)

In [6]:
from sklearn.metrics import pairwise_distances_argmin_min
import numpy as np
import pandas as pd

# نقاط نرمال (label = 0)
normal_points = scaled_df_clean[labels != -1]
# نقاط غیرعادی (label = -1)
anomaly_indices = scaled_df_clean.index[labels == -1]
anomalies = scaled_df_clean.loc[anomaly_indices]


# محاسبه مرکز ثقل خوشه‌ها
unique_labels = set(labels) - {-1}
centroids = np.array([scaled_df_clean[labels == label].mean(axis=0) for label in unique_labels])

# محاسبه فاصله هر anomaly از نزدیک‌ترین مرکز خوشه
_, distances = pairwise_distances_argmin_min(anomalies, centroids)

# انتخاب مهم‌ترین موارد غیرعادی بر اساس بیشترین فاصله
top_n = 10
top_anomalies_idx_in_anomalies = distances.argsort()[::-1][:top_n]
important_anomalies = anomalies.iloc[top_anomalies_idx_in_anomalies]

# بازگردانی مقادیر اصلی از حالت نرمال‌شده
important_anomalies_original = scaler.inverse_transform(important_anomalies)

# تبدیل به دیتافریم با ستون‌های اصلی و اضافه کردن فاصله از مرکز خوشه
important_anomalies_df = pd.DataFrame(
    important_anomalies_original,
    columns=scaled_df_clean.columns, # Use scaled_df_clean.columns as anomalies is derived from it
    index=important_anomalies.index
)
important_anomalies_df['distance_from_centroid'] = distances[top_anomalies_idx_in_anomalies]


In [7]:
from sklearn.metrics import pairwise_distances_argmin_min
import numpy as np
import pandas as pd

# استخراج نقاط غیرعادی
anomalies = scaled_df_clean[labels == -1]

# اضافه کردن برچسب خوشه به دیتافریم استانداردشده
scaled_df_clean['dbscan_label'] = labels

# انتخاب فقط ستون‌های عددی برای محاسبه مرکز ثقل
numeric_cols = scaled_df_clean.select_dtypes(include=np.number).columns.tolist()
# حذف ستون های برچسب گذاری که عددی هستند اما برای خوشه بندی استفاده نشده اند
if 'dbscan_label' in numeric_cols:
    numeric_cols.remove('dbscan_label')
if 'anomaly_label' in numeric_cols:
    numeric_cols.remove('anomaly_label')


# محاسبه مرکز ثقل خوشه‌ها (بدون در نظر گرفتن نقاط غیرعادی)
unique_labels = set(labels) - {-1}
centroids = np.array([scaled_df_clean[scaled_df_clean['dbscan_label'] == label][numeric_cols].mean(axis=0) for label in unique_labels])


# محاسبه فاصله هر anomaly از نزدیک‌ترین مرکز خوشه
# اطمینان از اینکه anomalies فقط شامل ستون‌های عددی است
anomalies_numeric = anomalies[numeric_cols]
_, distances = pairwise_distances_argmin_min(anomalies_numeric, centroids)


# مرتب‌سازی بر اساس فاصله (نزولی)
sorted_idx = distances.argsort()[::-1]
sorted_anomalies_scaled = anomalies.iloc[sorted_idx]
sorted_distances = distances[sorted_idx]

# بازگردانی مقادیر اصلی از حالت نرمال‌شده
# اطمینان از اینکه anomalies_original فقط شامل ستون‌های عددی است
anomalies_original_numeric = scaler.inverse_transform(anomalies[numeric_cols])

# ساخت دیتافریم نهایی با مقادیر اصلی و فاصله از مرکز خوشه
anomalies_df = pd.DataFrame(
    anomalies_original_numeric,
    columns=numeric_cols, # استفاده از نام ستون های عددی
    index=sorted_anomalies_scaled.index # استفاده از ایندکس نمونه‌های مرتب شده
)
anomalies_df['distance_from_centroid'] = sorted_distances


In [8]:
# اجرای DBSCAN روی داده‌های استانداردشده
dbscan = DBSCAN(eps=0.6, min_samples=9) # Using the best params found earlier
labels = dbscan.fit_predict(scaled_df_clean)

# اضافه کردن برچسب خوشه به دیتافریم استانداردشده
scaled_df_clean['dbscan_label'] = labels

# افزودن برچسب DBSCAN به دیتافریم اصلی با استفاده از ایندکس
df1 = df1.merge(scaled_df_clean[['dbscan_label']], left_index=True, right_index=True, how='left')

# فیلتر کردن نمونه‌های غیرنرمال (برچسب -1)
abnormal_df = df1[df1['dbscan_label'] == -1]

# انتخاب ۱۰ نمونه تصادفی از موارد غیرنرمال
abnormal_sample = abnormal_df.sample(n=10, random_state=42).reset_index(drop=True)


In [9]:
from sklearn.cluster import DBSCAN
import pandas as pd

# اجرای DBSCAN روی داده‌های استانداردشده
dbscan = DBSCAN(eps=0.6, min_samples=9)
labels = dbscan.fit_predict(scaled_df_clean)

# افزودن برچسب خوشه به دیتافریم استانداردشده
scaled_df_clean['dbscan_label'] = labels

# افزودن برچسب DBSCAN به دیتافریم اصلی با استفاده از ایندکس
df1 = df1.merge(scaled_df_clean[['dbscan_label']], left_index=True, right_index=True, how='left', suffixes=('', '_y'))

# حذف ستون‌های تکراری در صورت وجود
df1 = df1.loc[:, ~df1.columns.duplicated()]

# فیلتر کردن نمونه‌های غیرنرمال (برچسب -1)
abnormal_df = df1[df1['dbscan_label'] == -1]

# انتخاب ۱۰ نمونه تصادفی از موارد غیرنرمال
abnormal_sample = abnormal_df.sample(n=10, random_state=42).reset_index(drop=True)

# حذف ستون برچسب DBSCAN
abnormal_sample = abnormal_sample.drop(columns=['dbscan_label'])


In [10]:
from sklearn.metrics import pairwise_distances

# نقاط نرمال (خوشه‌ای)
normal_points = scaled_df_clean[labels != -1]
# نقاط غیرعادی
outliers = scaled_df_clean[labels == -1]

# فاصله هر نقطه غیرعادی تا نزدیک‌ترین نقطه نرمال
distances = pairwise_distances(outliers, normal_points)
min_distances = distances.min(axis=1)


In [11]:
# ساخت DataFrame برای نقاط غیرعادی و فاصله‌ها
outlier_df = pd.DataFrame(outliers)
outlier_df["min_distance_to_cluster"] = min_distances

# انتخاب تصادفی ۱۰ نقطه
np.random.seed(42)  # برای reproducibility
sampled_outliers = outlier_df.sample(n=10)

# مرتب‌سازی بر اساس فاصله (شدت غیرعادی بودن)
sampled_outliers_sorted = sampled_outliers.sort_values(by="min_distance_to_cluster", ascending=False)


In [12]:
from sklearn.neighbors import LocalOutlierFactor

lof = LocalOutlierFactor(n_neighbors=20, novelty=False)
lof_scores = -lof.fit_predict(outliers)  # نمره LOF (بزرگ‌تر = غیرعادی‌تر)
lof_factors = -lof.negative_outlier_factor_

In [13]:
# ساخت DataFrame برای نقاط نویز و نمره LOF
lof_df = pd.DataFrame(outliers)
lof_df["LOF_score"] = lof_factors  # مقدار LOF (بزرگ‌تر = غیرعادی‌تر)

# انتخاب تصادفی ۱۰ نقطه
np.random.seed(42)  # برای reproducibility
sampled_lof = lof_df.sample(n=10)

# مرتب‌سازی بر اساس نمره LOF
sampled_lof_sorted = sampled_lof.sort_values(by="LOF_score", ascending=False)


In [14]:
# انتخاب تعداد همسایه‌ها (k)
k = 10

# مدل همسایگی روی نقاط غیرعادی
nbrs = NearestNeighbors(n_neighbors=k)
nbrs.fit(outliers)

# محاسبه فاصله‌ها تا k همسایه
distances, indices = nbrs.kneighbors(outliers)

# فاصله تا k‌امین همسایه (یعنی فاصله کم‌تراکم‌ترین نقطه اطراف)
k_distances = distances[:, -1]

# ساخت DataFrame برای تحلیل
density_df = pd.DataFrame(outliers)
density_df["k_distance"] = k_distances

# انتخاب تصادفی ۱۰ نقطه
np.random.seed(42)
sampled_density = density_df.sample(n=10)

# مرتب‌سازی بر اساس فاصله (کم‌تراکم‌ترین اول)
sampled_density_sorted = sampled_density.sort_values(by="k_distance", ascending=False)


In [15]:
from sklearn.preprocessing import MinMaxScaler

# نقاط نرمال و غیرعادی
normal_points = scaled_df_clean[labels != -1]
outliers = scaled_df_clean[labels == -1]

# 1. فاصله از نزدیک‌ترین نقطه نرمال
distances = pairwise_distances(outliers, normal_points)
min_distances = distances.min(axis=1)

# 2. نمره LOF
lof = LocalOutlierFactor(n_neighbors=20, novelty=False)
lof.fit(outliers)
lof_scores = -lof.negative_outlier_factor_

# 3. k-distance (تراکم محلی)
k = 10
nbrs = NearestNeighbors(n_neighbors=k)
nbrs.fit(outliers)
k_distances = nbrs.kneighbors(outliers)[0][:, -1]

# ساخت DataFrame
ensemble_df = pd.DataFrame(outliers)
ensemble_df["distance_to_cluster"] = min_distances
ensemble_df["lof_score"] = lof_scores
ensemble_df["k_distance"] = k_distances

# نرمال‌سازی هر شاخص بین 0 و 1
scaler = MinMaxScaler()
normalized = scaler.fit_transform(ensemble_df[["distance_to_cluster", "lof_score", "k_distance"]])
ensemble_df[["norm_dist", "norm_lof", "norm_kdist"]] = normalized

# محاسبه نمره ترکیبی
ensemble_df["ensemble_score"] = ensemble_df[["norm_dist", "norm_lof", "norm_kdist"]].mean(axis=1)

# انتخاب تصادفی ۱۰ نقطه و مرتب‌سازی بر اساس نمره ترکیبی
np.random.seed(42)
sampled = ensemble_df.sample(n=10)
sampled_sorted = sampled.sort_values(by="ensemble_score", ascending=False)
